# 07 - Final Models and Hyperparameter Tuning

## Objective

This notebook performs hyperparameter tuning for the two final classifiers:

- Random Forest
- XGBoost

The proposed feature set uses:
- Autoencoder-based feature selection at the **90% cumulative-importance threshold**
- K-Means cluster information using the selected cluster count from the previous notebook

Hyperparameter tuning is performed with cross-validation on the training data only. The test set remains unseen until the final evaluation.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.neural_network import MLPRegressor
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from scipy.stats import randint, uniform

np.random.seed(42)

print("Libraries imported successfully.")


Libraries imported successfully.


In [2]:
# Load the dataset
df = pd.read_csv("../data/diabetes_dataset.csv")

print("Dataset shape:", df.shape)


Dataset shape: (100000, 31)


In [3]:
# Prepare target and leakage-free input features
target_column = "diagnosed_diabetes"
leakage_columns = ["diabetes_stage"]

X = df.drop(columns=[target_column] + leakage_columns)
y = df[target_column].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])
print("Input features before encoding:", X.shape[1])


Training samples: 80000
Testing samples: 20000
Input features before encoding: 29


In [4]:
# Preprocess the data
numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)


Processed training shape: (80000, 47)
Processed testing shape: (20000, 47)


## Reproduce the 90% Autoencoder Feature Selection

The Autoencoder is trained without the target variable, using the same architecture and random seed as Notebook 05. The 90% cumulative-importance threshold is then applied.


In [5]:
# Autoencoder
autoencoder = MLPRegressor(
    hidden_layer_sizes=(32, 12, 32),
    activation="relu",
    solver="adam",
    batch_size=256,
    learning_rate_init=0.001,
    max_iter=50,
    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=5,
    random_state=42
)

autoencoder.fit(X_train_processed, X_train_processed)

encoder_weights = autoencoder.coefs_[0]
raw_importance = np.sum(np.abs(encoder_weights), axis=1)
importance = raw_importance / raw_importance.sum()

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance
}).sort_values("Importance", ascending=False).reset_index(drop=True)

feature_importance["Cumulative_Percent"] = (
    feature_importance["Importance"].cumsum() * 100
)

first_index = feature_importance.index[
    feature_importance["Cumulative_Percent"] >= 90
][0]

selected_features = feature_importance.loc[:first_index, "Feature"].tolist()
selected_indices = [list(feature_names).index(f) for f in selected_features]

X_train_selected = X_train_processed[:, selected_indices]
X_test_selected = X_test_processed[:, selected_indices]

print("90% selected features:", len(selected_features))
print(
    "Actual cumulative importance:",
    round(feature_importance.loc[first_index, "Cumulative_Percent"], 4),
    "%"
)


90% selected features: 36
Actual cumulative importance: 90.0169 %


## Recreate K-Means and Add Cluster Information

We select K using the highest silhouette score for K=2..6. The final K-Means model is fitted on the entire training set after K selection.


In [6]:
# Use a sample for selecting K
sample_size = min(10000, X_train_selected.shape[0])
rng = np.random.RandomState(42)

sample_indices = rng.choice(
    X_train_selected.shape[0],
    size=sample_size,
    replace=False
)

X_cluster_sample = X_train_selected[sample_indices]

k_values = range(2, 7)
silhouette_scores = []
inertias = []

for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_cluster_sample)
    inertias.append(model.inertia_)

    from sklearn.metrics import silhouette_score
    silhouette_scores.append(
        silhouette_score(X_cluster_sample, labels)
    )

cluster_eval = pd.DataFrame({
    "K": list(k_values),
    "Inertia": inertias,
    "Silhouette_Score": silhouette_scores
})

best_k = int(
    cluster_eval.loc[
        cluster_eval["Silhouette_Score"].idxmax(), "K"
    ]
)

print(cluster_eval.round(4))
print("Selected K:", best_k)


   K      Inertia  Silhouette_Score
0  2  141640.2585            0.1377
1  3  130826.9391            0.0942
2  4  122546.7800            0.1018
3  5  117750.6449            0.0935
4  6  115016.7714            0.0901
Selected K: 2


In [7]:
# Fit final K-Means on all training data
kmeans_final = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

train_clusters = kmeans_final.fit_predict(X_train_selected)
test_clusters = kmeans_final.predict(X_test_selected)

X_train_final = np.column_stack([
    X_train_selected,
    train_clusters
])

X_test_final = np.column_stack([
    X_test_selected,
    test_clusters
])

print("Final training shape:", X_train_final.shape)
print("Final testing shape:", X_test_final.shape)
print("Cluster feature added:", best_k, "clusters")


Final training shape: (80000, 37)
Final testing shape: (20000, 37)
Cluster feature added: 2 clusters


## Baseline Reference

The baseline results obtained earlier (before Autoencoder feature selection and clustering) were:

- Random Forest: Accuracy 0.9198, ROC-AUC 0.9424
- XGBoost: Accuracy 0.9193, ROC-AUC 0.9423

The purpose of tuning below is to determine whether optimized hyperparameters improve the proposed final approach.


## 7.1 Random Forest Hyperparameter Tuning

RandomizedSearchCV is used instead of an exhaustive grid so that a meaningful search can be performed on the 80,000 training records without testing every possible combination.


In [8]:
rf_base = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

rf_param_distributions = {
    "n_estimators": randint(150, 401),
    "max_depth": [None, 8, 12, 16, 20, 24],
    "min_samples_split": randint(2, 11),
    "min_samples_leaf": randint(1, 6),
    "max_features": ["sqrt", "log2", None],
    "class_weight": [None, "balanced"]
}

rf_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=rf_param_distributions,
    n_iter=8,
    scoring="roc_auc",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

rf_search.fit(X_train_final, y_train)

print("Best Random Forest parameters:")
print(rf_search.best_params_)
print("Best cross-validation ROC-AUC:",
      round(rf_search.best_score_, 4))


Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best Random Forest parameters:
{'class_weight': None, 'max_depth': 8, 'max_features': None, 'min_samples_leaf': 3, 'min_samples_split': 9, 'n_estimators': 266}
Best cross-validation ROC-AUC: 0.9437


In [9]:
# Evaluate tuned Random Forest on the unseen test set
rf_tuned = rf_search.best_estimator_

rf_pred = rf_tuned.predict(X_test_final)
rf_prob = rf_tuned.predict_proba(X_test_final)[:, 1]

rf_tuned_results = {
    "Model": "Tuned Random Forest - 90% + Cluster",
    "Accuracy": accuracy_score(y_test, rf_pred),
    "Precision": precision_score(y_test, rf_pred),
    "Recall": recall_score(y_test, rf_pred),
    "F1-Score": f1_score(y_test, rf_pred),
    "ROC-AUC": roc_auc_score(y_test, rf_prob)
}

pd.DataFrame([rf_tuned_results]).round(4)


,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Tuned Random Forest - 90% + Cluster,0.9102,0.9992,0.851,0.9192,0.9416


## 7.2 XGBoost Hyperparameter Tuning

In [10]:
xgb_base = XGBClassifier(
    random_state=42,
    n_jobs=-1,
    eval_metric="logloss"
)

xgb_param_distributions = {
    "n_estimators": randint(100, 401),
    "max_depth": randint(3, 9),
    "learning_rate": uniform(0.03, 0.17),
    "subsample": uniform(0.7, 0.3),
    "colsample_bytree": uniform(0.7, 0.3),
    "min_child_weight": randint(1, 8),
    "gamma": uniform(0, 0.5)
}

xgb_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=xgb_param_distributions,
    n_iter=8,
    scoring="roc_auc",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

xgb_search.fit(X_train_final, y_train)

print("Best XGBoost parameters:")
print(xgb_search.best_params_)
print("Best cross-validation ROC-AUC:",
      round(xgb_search.best_score_, 4))


Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best XGBoost parameters:
{'colsample_bytree': np.float64(0.935552788417904), 'gamma': np.float64(0.09983689107917987), 'learning_rate': np.float64(0.11741985453031398), 'max_depth': 3, 'min_child_weight': 3, 'n_estimators': 234, 'subsample': np.float64(0.7511572371061874)}
Best cross-validation ROC-AUC: 0.942


In [11]:
# Evaluate tuned XGBoost on the unseen test set
xgb_tuned = xgb_search.best_estimator_

xgb_pred = xgb_tuned.predict(X_test_final)
xgb_prob = xgb_tuned.predict_proba(X_test_final)[:, 1]

xgb_tuned_results = {
    "Model": "Tuned XGBoost - 90% + Cluster",
    "Accuracy": accuracy_score(y_test, xgb_pred),
    "Precision": precision_score(y_test, xgb_pred),
    "Recall": recall_score(y_test, xgb_pred),
    "F1-Score": f1_score(y_test, xgb_pred),
    "ROC-AUC": roc_auc_score(y_test, xgb_prob)
}

pd.DataFrame([xgb_tuned_results]).round(4)


,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Tuned XGBoost - 90% + Cluster,0.9102,0.9983,0.8518,0.9193,0.9404


## 7.3 Tuned Model Comparison

In [12]:
tuned_results = pd.DataFrame([
    rf_tuned_results,
    xgb_tuned_results
])

tuned_results.round(4)


,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Tuned Random Forest - 90% + Cluster,0.9102,0.9992,0.8510,0.9192,0.9416
1,Tuned XGBoost - 90% + Cluster,0.9102,0.9983,0.8518,0.9193,0.9404


## 7.4 Before vs After Tuning

The final notebook should compare the tuned results against the untuned 90% + cluster results from Notebook 06.

The comparison should be based on ROC-AUC, F1-score, Recall, Precision, and Accuracy, not accuracy alone.


In [13]:
# Untuned 90% + cluster reference values from Notebook 06
untuned_results = pd.DataFrame([
    {
        "Model": "Random Forest - 90% + Cluster",
        "Accuracy": 0.9092,
        "Precision": 0.9961,
        "Recall": 0.8519,
        "F1-Score": 0.9184,
        "ROC-AUC": 0.9406
    },
    {
        "Model": "XGBoost - 90% + Cluster",
        "Accuracy": 0.9100,
        "Precision": 0.9965,
        "Recall": 0.8531,
        "F1-Score": 0.9192,
        "ROC-AUC": 0.9391
    }
])

before_after = pd.concat([
    untuned_results.assign(Stage="Before Tuning"),
    tuned_results.assign(Stage="After Tuning")
], ignore_index=True)

before_after.round(4)


,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC,Stage
0,Random Forest - 90% + Cluster,0.9092,0.9961,0.8519,0.9184,0.9406,Before Tuning
1,XGBoost - 90% + Cluster,0.9100,0.9965,0.8531,0.9192,0.9391,Before Tuning
2,Tuned Random Forest - 90% + Cluster,0.9102,0.9992,0.8510,0.9192,0.9416,After Tuning
3,Tuned XGBoost - 90% + Cluster,0.9102,0.9983,0.8518,0.9193,0.9404,After Tuning


## Tuning Findings

Record:

1. Search method: RandomizedSearchCV.
2. Cross-validation: 3-fold.
3. Parameters considered.
4. Best Random Forest parameters.
5. Best XGBoost parameters.
6. Best cross-validation ROC-AUC for each model.
7. Test-set performance before and after tuning.
8. Whether tuning improved the proposed approach.
